# CNN-VAE Effective Latent Dimensionality

Run this notebook from the `cnn_vae/` directory. It scans trained CNN-VAE checkpoints, collects encoder means `mu(x)` for raw jet events, centers the latent cloud, and computes a PCA spectrum to estimate how many latent directions are effectively used.

In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path.cwd()
CODE_DIR = PROJECT_ROOT / "code_training"
sys.path.insert(0, str(CODE_DIR))

from betaVae import BetaVAE
from jet_image_utils import load_events_from_dat, event_to_jet_image, event_to_jet_image_no_pid, pt_normalize, build_pid_channel_map, build_pid_lookup

plt.rcParams["figure.dpi"] = 120
torch.set_grad_enabled(False)
PROJECT_ROOT

## Configuration

In [ ]:
DATA_PATH = PROJECT_ROOT.parent / "data" / "u_jet_real_sample.dat"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "u_jet_real_sample.dat"

CHECKPOINT_GLOB = "results/200GeVjet_nopid_annealing/mse_all/*/best_model_betavae.pth"
OUTPUT_DIR = PROJECT_ROOT / "results" / "200GeVjet_nopid_annealing/mse_all" / "latent_effective_dim_cnn"

BATCH_SIZE = 512
NUM_WORKERS = 0
MAX_EVENTS = None  # set an integer for a faster quick test, e.g. 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("data      :", DATA_PATH)
print("device    :", DEVICE)
print("output dir:", OUTPUT_DIR)

In [ ]:
checkpoint_paths = sorted(p for p in PROJECT_ROOT.glob(CHECKPOINT_GLOB) if p.is_file())
if not checkpoint_paths:
    raise FileNotFoundError(f"No checkpoints matched {CHECKPOINT_GLOB!r}")

pd.DataFrame({"checkpoint": [str(p.relative_to(PROJECT_ROOT)) for p in checkpoint_paths]})

## Helper functions

In [ ]:
class JetImageDataset(Dataset):
    def __init__(self, events, pid_lookup, pid_min, n_channels, img_size, checkpoint):
        self.events = events
        self.pid_lookup = pid_lookup
        self.pid_min = pid_min
        self.n_channels = n_channels
        self.img_size = img_size
        self.checkpoint = checkpoint

    def __len__(self):
        return len(self.events)

    def __getitem__(self, idx):
        if checkpoint_uses_no_pid_images(self.checkpoint):
            img = event_to_jet_image_no_pid(
                self.events[idx],
                eta_bins=self.img_size, phi_bins=self.img_size,
                eta_max=0.5, phi_max=0.5,
            )
        else:
            img = event_to_jet_image(
                self.events[idx], self.pid_lookup, self.pid_min, self.n_channels,
                eta_bins=self.img_size, phi_bins=self.img_size,
                eta_max=0.5, phi_max=0.5,
            )
        img = pt_normalize(img).astype(np.float32)
        if checkpoint_uses_standardized_images(self.checkpoint):
            mean_img = np.asarray(self.checkpoint["mean_img"], dtype=np.float32)
            std_img = np.asarray(self.checkpoint["std_img"], dtype=np.float32)
            img = ((img - mean_img) / (std_img + 1e-5)).astype(np.float32)
        return torch.from_numpy(img)


def pid_map_from_checkpoint_or_events(checkpoint, events):
    if "pid_channel_map" in checkpoint:
        return {int(pid): int(ch) for pid, ch in checkpoint["pid_channel_map"].items()}
    return build_pid_channel_map(events)


def checkpoint_hidden_dims(checkpoint):
    return list(checkpoint.get("hidden_dims", [32, 64, 128, 256, 512]))


def checkpoint_output_mode(checkpoint):
    if "output_mode" in checkpoint:
        return str(checkpoint["output_mode"])
    if checkpoint.get("normalization", None) == "event_pt":
        return "event_pt"
    return "tanh"


def checkpoint_channel_mode(checkpoint):
    if "channel_mode" in checkpoint:
        return str(checkpoint["channel_mode"])
    pid_map = checkpoint.get("pid_channel_map", {})
    pid_map = {int(pid): int(ch) for pid, ch in pid_map.items()} if hasattr(pid_map, "items") else {}
    if int(checkpoint["n_channels"]) == 1 and (not pid_map or pid_map == {0: 0}):
        return "eta_phi_total_pt_no_pid"
    return "pid_channels"


def checkpoint_uses_no_pid_images(checkpoint):
    return checkpoint_channel_mode(checkpoint) == "eta_phi_total_pt_no_pid"


def checkpoint_uses_standardized_images(checkpoint):
    return "mean_img" in checkpoint and "std_img" in checkpoint and checkpoint.get("normalization", None) != "event_pt"


def load_model_from_checkpoint(checkpoint, device):
    model = BetaVAE(
        in_channels=int(checkpoint["n_channels"]),
        latent_dim=int(checkpoint["latent_dim"]),
        image_size=int(checkpoint["img_size"]),
        hidden_dims=checkpoint_hidden_dims(checkpoint),
        output_mode=checkpoint_output_mode(checkpoint),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def collect_latent_mu(model, dataset, batch_size, num_workers, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    chunks = []
    for batch in loader:
        x = batch.to(device)
        mu, _ = model.encode(x)
        chunks.append(mu.detach().cpu().numpy())
    return np.concatenate(chunks, axis=0)

In [ ]:
def pca_spectrum(mu):
    centered = mu - mu.mean(axis=0, keepdims=True)
    _, s, vt = np.linalg.svd(centered, full_matrices=False)
    denom = max(centered.shape[0] - 1, 1)
    eigenvalues = (s ** 2) / denom
    total = eigenvalues.sum()
    explained_ratio = np.zeros_like(eigenvalues) if total <= 0 else eigenvalues / total
    cumulative = np.cumsum(explained_ratio)
    scores = centered @ vt[:2].T if vt.shape[0] >= 2 else centered @ vt.T
    return eigenvalues, explained_ratio, cumulative, scores


def effective_dim(cumulative, threshold):
    if cumulative.size == 0 or cumulative[-1] <= 0:
        return 0
    return int(np.searchsorted(cumulative, threshold) + 1)


def participation_ratio(eigenvalues):
    s1 = float(eigenvalues.sum())
    s2 = float(np.sum(eigenvalues ** 2))
    return 0.0 if s2 <= 0 else (s1 ** 2) / s2


def correlation_pca_participation_ratio(mu):
    centered = mu - mu.mean(axis=0, keepdims=True)
    std = centered.std(axis=0, ddof=0)
    valid = std > 0.0
    if valid.sum() == 0:
        return 0.0
    if valid.sum() == 1:
        return 1.0
    standardized = centered[:, valid] / std[valid]
    corr = np.corrcoef(standardized, rowvar=False)
    corr_eigenvalues = np.linalg.eigvalsh(corr)
    corr_eigenvalues = np.clip(corr_eigenvalues, 0.0, None)
    return participation_ratio(corr_eigenvalues)


def checkpoint_label(path, checkpoint):
    parent = path.parent.name
    m = re.search(r"beta_([0-9p]+)", parent)
    beta = m.group(1).replace("p", ".") if m else str(checkpoint.get("beta", "?"))
    return f"beta={beta}"

## Load events

In [ ]:
events = load_events_from_dat(str(DATA_PATH))
if MAX_EVENTS is not None:
    events = events[:MAX_EVENTS]
print("events:", len(events))

## Run PCA analysis

In [ ]:
summary_rows = []
spectra = {}

for path in checkpoint_paths:
    print(f"Analyzing {path.relative_to(PROJECT_ROOT)}")
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = load_model_from_checkpoint(checkpoint, DEVICE)

    pid_channel_map = pid_map_from_checkpoint_or_events(checkpoint, events)
    pid_lookup, pid_min = build_pid_lookup(pid_channel_map)
    dataset = JetImageDataset(
        events,
        pid_lookup=pid_lookup,
        pid_min=pid_min,
        n_channels=int(checkpoint["n_channels"]),
        img_size=int(checkpoint["img_size"]),
        checkpoint=checkpoint,
    )

    mu = collect_latent_mu(model, dataset, BATCH_SIZE, NUM_WORKERS, DEVICE)
    eigenvalues, explained_ratio, cumulative, scores = pca_spectrum(mu)
    label = checkpoint_label(path, checkpoint)

    spectra[label] = {
        "checkpoint": str(path.relative_to(PROJECT_ROOT)),
        "eigenvalues": eigenvalues,
        "explained_ratio": explained_ratio,
        "cumulative": cumulative,
        "scores_2d": scores[:, :2],
    }
    summary_rows.append({
        "label": label,
        "checkpoint": str(path.relative_to(PROJECT_ROOT)),
        "latent_dim": int(checkpoint["latent_dim"]),
        "img_size": int(checkpoint["img_size"]),
        "hidden_dims": ",".join(map(str, checkpoint_hidden_dims(checkpoint))),
        "output_mode": checkpoint_output_mode(checkpoint),
        "channel_mode": checkpoint_channel_mode(checkpoint),
        "legacy_standardized_input": checkpoint_uses_standardized_images(checkpoint),
        "val_loss": float(checkpoint.get("val_loss", np.nan)),
        "d_eff_90": effective_dim(cumulative, 0.90),
        "d_eff_95": effective_dim(cumulative, 0.95),
        "d_eff_99": effective_dim(cumulative, 0.99),
        "participation_ratio": correlation_pca_participation_ratio(mu),
        "latent_variance_sum": float(eigenvalues.sum()),
    })

summary = pd.DataFrame(summary_rows).sort_values("checkpoint").reset_index(drop=True)
summary_path = OUTPUT_DIR / "latent_effective_dim_summary.csv"
summary.to_csv(summary_path, index=False)
summary

## Cumulative explained variance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, spec in spectra.items():
    k = np.arange(1, len(spec["cumulative"]) + 1)
    ax.plot(k, spec["cumulative"], marker="o", linewidth=1.5, markersize=3, label=label)

for threshold in [0.90, 0.95, 0.99]:
    ax.axhline(threshold, color="0.75", linestyle="--", linewidth=1)
    ax.text(1, threshold + 0.005, f"{threshold:.0%}", color="0.35", fontsize=9)

ax.set_xlabel("number of principal components")
ax.set_ylabel("cumulative explained variance")
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()

cumulative_path = OUTPUT_DIR / "pca_cumulative_explained_variance.png"
fig.savefig(cumulative_path, dpi=160)
plt.show()

## Scree plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, spec in spectra.items():
    k = np.arange(1, len(spec["explained_ratio"]) + 1)
    ax.plot(k, spec["explained_ratio"], marker="o", linewidth=1.5, markersize=3, label=label)

ax.set_xlabel("principal component")
ax.set_ylabel("explained variance ratio")
ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=8)
fig.tight_layout()

scree_path = OUTPUT_DIR / "pca_scree_explained_ratio.png"
fig.savefig(scree_path, dpi=160)
plt.show()

## Effective dimension summary

In [ ]:
plot_df = summary.copy()
plot_df["x_label"] = plot_df["label"]

fig, ax = plt.subplots(figsize=(max(7, 1.2 * len(plot_df)), 5))
x = np.arange(len(plot_df))
width = 0.25
ax.bar(x - width, plot_df["d_eff_90"], width, label="90%")
ax.bar(x, plot_df["d_eff_95"], width, label="95%")
ax.bar(x + width, plot_df["d_eff_99"], width, label="99%")
ax.plot(x, plot_df["participation_ratio"], color="black", marker="o", linewidth=1.5, label="correlation PCA participation ratio")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["x_label"], rotation=35, ha="right")
ax.set_ylabel("effective latent dimension")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()

bar_path = OUTPUT_DIR / "effective_dimension_by_checkpoint.png"
fig.savefig(bar_path, dpi=160)
plt.show()

## Optional PC1-PC2 scatter

In [ ]:
n_models = len(spectra)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4), squeeze=False)

for ax, (label, spec) in zip(axes.ravel(), spectra.items()):
    scores = spec["scores_2d"]
    if scores.shape[1] < 2:
        ax.text(0.5, 0.5, "latent_dim < 2", ha="center", va="center")
        ax.set_axis_off()
        continue
    ax.scatter(scores[:, 0], scores[:, 1], s=4, alpha=0.35)
    ax.set_title(label)
    ax.set_xlabel("PC1 score")
    ax.set_ylabel("PC2 score")
    ax.grid(True, alpha=0.25)

fig.tight_layout()
scatter_path = OUTPUT_DIR / "latent_pc1_pc2_scatter.png"
fig.savefig(scatter_path, dpi=160)
plt.show()

In [ ]:
print("Saved:")
print(summary_path)
print(cumulative_path)
print(scree_path)
print(bar_path)
print(scatter_path)